# Image Understanding Concepts

**Module:** 15 — VLMs & Multimodal

Captioning, VQA, grounding, tagging, and visual prompting — the task taxonomy behind multimodal apps.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Distinguish captioning, classification, VQA, grounding, and scene graphs
- Write visual prompts that reduce hallucination
- Design lightweight evaluation for image understanding
- Validate grounding geometry and build tiny scene graphs


## Core Tasks

### Definition
Image understanding maps pixels (+ optional text) to labels, language, regions, or graphs.

### Why it matters
Wrong task framing wastes capacity — open VQA when you needed closed ontology tags.

### How it works
Start from the output contract (string, enum, bbox, JSON), then pick task + schema.

### Intuition
Captioning narrates; VQA answers; grounding points; graphs relate.

### Pitfalls
- Open essays for machine fields
- No ontology → tag drift
- 'What do you see?' for compliance extraction

### When to use
Any product turning images into searchable or actionable structure.


### Task comparison

| Task | Prompt shape | Output | Eval |
|------|--------------|--------|------|
| Captioning | Describe… | Text | CIDEr / Likert / judge |
| Classification | Pick one of… | Label | F1 |
| VQA | Question | Short answer | Exact/soft |
| Grounding | Referring expression | BBox | IoU |
| Scene graph | List relations | Nodes+edges | Graph edit |

```mermaid
flowchart TB
  IMG[Image] --> CAP[Caption]
  IMG --> TAG[Tags]
  IMG --> VQA[VQA]
  IMG --> GR[Ground]
  CAP --> APP[Product surfaces]
  TAG --> APP
  VQA --> APP
  GR --> APP
```


In [ ]:
# Demo 1: prompt templates by task
TEMPLATES = {
    "caption": "Describe in 2 sentences. Mention readable text.",
    "tags": "Return ≤5 tags from ontology {onto}. JSON list only.",
    "vqa": "Use only visible evidence; else UNKNOWN.\nQ: {q}",
    "ground": "Locate '{phrase}' as bbox [x,y,w,h] normalized 0-1. JSON only.",
}
def build(task, **kw): return TEMPLATES[task].format(**kw)
print(build("vqa", q="How many helmets?"))
print(build("tags", onto="helmet,vest,cone"))
print(build("ground", phrase="leftmost cone"))


In [ ]:
# Demo 2: exact vs soft VQA scoring
from difflib import SequenceMatcher

def soft(pred, gold, thr=0.6):
    p, g = pred.lower().strip(), gold.lower().strip()
    if p == g or g in p or p in g: return True
    return SequenceMatcher(None, p, g).ratio() >= thr

pairs = [("red","red"),("2","two"),("stop sign","a stop sign"),("UNKNOWN","three")]
for pred, gold in [("red","red"),("2","two"),("stop sign","a stop sign on right"),("UNKNOWN","I think three")]:
    print(pred, gold, "soft=", soft(pred, gold))


## Captioning & Dense Description

### Definition
Captioning produces NL descriptions; dense/region captions attach phrases to areas.

### Why it matters
Accessibility and indexing start here — bad captions poison retrieval.

### How it works
Control length, style (alt-text vs marketing), and whether to read on-image text.

### Intuition
Good alt-text: what must a blind user know *now*?

### Pitfalls
- Poetic captions omitting critical text
- Listing every pixel vs missing the subject

### When to use
Alt-text, media search, first pass before structured extract.


In [ ]:
# Demo 3: caption styles
def stylize(facts, style):
    if style == "alt_text": return "Photo: " + "; ".join(facts[:3]) + "."
    if style == "dense": return " | ".join(f"{i+1}. {f}" for i,f in enumerate(facts))
    if style == "seo": return ", ".join(facts) + " — stock scene"
    raise ValueError(style)
facts = ["red bicycle on brick wall", "daylight", "sign reads OPEN"]
for s in ("alt_text","dense","seo"): print(s, "=>", stylize(facts, s))


## Grounding & Referring Expressions

### Definition
Grounding links a phrase to spatial evidence (box/mask/points).

### Why it matters
UI automation, robotics, and cited DocQA need *where*, not only *what*.

### How it works
Emit normalized boxes; validate bounds and area; disambiguate multiples.

### Intuition
Language points; geometry confirms.

### Pitfalls
- Pixel boxes without image size
- Ambiguous phrases with many matches

### When to use
Click-to-ask UIs, visual citation, computer-use agents.


In [ ]:
# Demo 4: bbox validation + IoU
def valid_bbox(b, min_area=0.001):
    if len(b)!=4: return False, "len"
    x,y,w,h = b
    if not all(0<=v<=1 for v in (x,y,w,h)): return False, "bounds"
    if x+w>1.01 or y+h>1.01: return False, "overflow"
    if w*h < min_area: return False, "tiny"
    return True, "ok"

def iou(a,b):
    ax2, ay2 = a[0]+a[2], a[1]+a[3]
    bx2, by2 = b[0]+b[2], b[1]+b[3]
    ix1,iy1 = max(a[0],b[0]), max(a[1],b[1])
    ix2,iy2 = min(ax2,bx2), min(ay2,by2)
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    union = a[2]*a[3] + b[2]*b[3] - inter
    return inter/union if union else 0.0

print(valid_bbox([0.1,0.2,0.3,0.25]), iou([0,0,0.5,0.5],[0.25,0.25,0.5,0.5]))


### Visual prompting tips

| Do | Don't |
|----|-------|
| Demand evidence / quote text | "Guess if unsure" |
| Constrain JSON schema | Open essays for fields |
| Crop ROI | Always send 40MP pages |
| Give ontology/units | Vague adjectives only |
| Separate perceive vs reason | One mega-prompt |

**Scene graphs:** nodes = objects/attributes; edges = relations (`cup —on→ table`).


In [ ]:
# Demo 5: scene graph from detections
from collections import defaultdict
dets = [{"id":"o1","label":"cup"},{"id":"o2","label":"table"},{"id":"o3","label":"book"}]
rels = [("o1","on","o2"),("o3","on","o2")]
idx = defaultdict(list)
for s,r,t in rels: idx[s].append(f"{r}->{t}")
print(dict(idx))


In [ ]:
# Demo 6: ontology-constrained tagger
ONTO = {"helmet","vest","vehicle","cone","sign"}
def tag(raw_tags):
    kept = [t.lower() for t in raw_tags if t.lower() in ONTO]
    unknown = [t for t in raw_tags if t.lower() not in ONTO]
    return {"tags": kept, "rejected": unknown}
print(tag(["Helmet","person","cone","banana"]))


### Evaluation mini-lab design

1. Freeze 50–200 images with gold answers per task.
2. Report exact, soft, and critical-error rate (safety/money fields).
3. Slice by: tiny text, counting, charts, low light.
4. Re-run on every model/prompt change (bakeoff discipline).


### Checklist — Prompt review

- [ ] Output format specified
- [ ] UNKNOWN allowed explicitly
- [ ] Ontology or units provided when needed
- [ ] Evidence instruction present
- [ ] Eval set exists for this prompt


### Try it yourself — Prompting & eval

1. Add a classification template that rejects out-of-ontology labels (done in spirit — extend Demo 6).
2. Write 5 adversarial VQA questions (tiny text, counting, color).
3. Compute soft-match accuracy on a 10-row mini benchmark.

**Stretch:** Unit-test IoU edge cases (no overlap, full containment).


### Try it yourself — Scene graphs

1. Add attributes to nodes and filter edges by label.
2. Export graph as Graphviz DOT string.


## Knowledge Check

**Q1.** When is grounding required over plain VQA?

<details><summary>Answer</summary>

When the product must show or act on *where* something is (citation, click, robot grasp).

</details>

**Q2.** Why prefer soft match + exact for VQA eval?

<details><summary>Answer</summary>

Exact is crisp for short answers; soft catches valid paraphrases like 2/two.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `VQA` | Visual question answering |
| `grounding` | Aligning phrases to regions |
| `ontology` | Controlled label vocabulary |
| `IoU` | Intersection-over-Union |
| `scene graph` | Objects + relations structure |
| `alt-text` | Accessibility image description |


## Key Takeaways

- Pick task from the output contract
- Prompts should demand evidence and constrain format
- Evaluate with task-appropriate metrics
- Grounding unlocks citation and UI automation


## Production Incident Patterns — image understanding

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Sudden cost spike | `detail=high` on huge pages | Resize + tile budget |
| Fluent wrong fields | VLM hallucination | OCR hybrid + schema |
| Cross-customer leak | Missing tenant filter | ACL in retriever code |
| Flaky eval scores | Unfrozen prompts/models | Pin versions + bakeoff set |
| Latency SLO burn | Full-page high detail | Crop ROI → mini model |

```
ASCII control loop:
  ingest -> normalize -> route model -> generate -> validate -> (HITL|export)
                     ^                              |
                     +-------- metrics/audit <------+
```


In [ ]:
# Cross-cutting: redact secrets before logging multimodal payloads
import re, json

SECRET_RE = re.compile(r"(api[_-]?key|bearer\s+[A-Za-z0-9._\-]+)", re.I)

def safe_log(payload: dict) -> str:
    s = json.dumps(payload)
    s = SECRET_RE.sub("***", s)
    if "base64," in s:
        s = re.sub(r"base64,[A-Za-z0-9+/=]+", "base64,[REDACTED]", s)
    return s[:500]

print(safe_log({
    "model": "gpt-4o",
    "api_key": "YOUR_OPENAI_API_KEY",
    "content": "data:image/png;base64,AAAABBBBCCCC",
    "topic": "image understanding",
}))


## Mini Case Study — image understanding

**Scenario:** A team ships a vision feature in one week. Demo looks great on three happy-path images.
**Week 2:** finance reports wrong totals; legal asks about image retention; GPU/API bill 4× forecast.

**Retro questions**
1. What was the output contract (schema) on day one?
2. Which failure mode had no metric?
3. Was there a crop/detail budget?
4. Who owns HITL and appeals?

**Design rule:** if a field can move money or identity, it needs a validator + disagreement path before automation.


In [ ]:
# Cross-cutting: simple SLO helper for vision endpoints
from dataclasses import dataclass

@dataclass
class VisionSLO:
    availability: float = 0.995
    p95_ms: int = 4000
    max_critical_field_error_rate: float = 0.005

def breached(slo: VisionSLO, avail: float, p95: int, crit_err: float) -> list[str]:
    out = []
    if avail < slo.availability: out.append("availability")
    if p95 > slo.p95_ms: out.append("latency")
    if crit_err > slo.max_critical_field_error_rate: out.append("critical_accuracy")
    return out or ["ok"]

print("image understanding", breached(VisionSLO(), 0.99, 5200, 0.02))


### Try it yourself — image understanding ops

1. Write a one-page runbook section for on-call when image understanding critical_accuracy SLO breaches.
2. Add a dashboard sketch: cost/1k images, CER/field error, HITL rate, p95 latency.
